In [1]:

import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

In [2]:
sales = pd.read_csv("../data/raw/sales_train_validation.csv")
calendar = pd.read_csv("../data/raw/calendar.csv")
price = pd.read_csv("../data/raw/sell_prices.csv")

In [3]:
sales_long = sales.melt(
    id_vars=[
        "id",
        "item_id",
        "dept_id",
        "cat_id",
        "store_id",
        "state_id"
    ],
    var_name="day",
    value_name="sales"
)

In [4]:
# Merge calendar information into sales dataset
# This adds dates, event names, SNAP days, weekdays, etc.

sales_long = sales_long.merge(
    calendar,
    left_on="day",
    right_on="d",
    how="left"
)

In [5]:
# Merge price data into the main dataset
# Matching is done using:
# - store_id
# - item_id
# - wm_yr_wk (week identifier)

sales_long = sales_long.merge(
    price,
    on=["store_id", "item_id", "wm_yr_wk"],
    how="left"
)

In [6]:
# Convert date column to datetime format
# This enables time-series operations and feature extraction

sales_long["date"] = pd.to_datetime(sales_long["date"])

In [7]:
# Check missing values in the dataset
# Sort columns by highest missing count

sales_long.isnull().sum().sort_values(ascending=False).head(20)

event_type_2    58205410
event_name_2    58205410
event_type_1    53631910
event_name_1    53631910
sell_price      12299413
month                  0
snap_WI                0
snap_TX                0
snap_CA                0
d                      0
year                   0
id                     0
item_id                0
weekday                0
wm_yr_wk               0
date                   0
sales                  0
day                    0
state_id               0
store_id               0
dtype: int64

In [8]:
# Fill missing prices using:
# 1. Forward fill (previous known value)
# 2. Backward fill (next known value)

sales_long["sell_price"] = (
    sales_long.groupby(["item_id", "store_id"])["sell_price"]
    .ffill()
    .bfill()
)

In [9]:
# Define event-related columns

event_cols = [
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

# Replace missing event values with "No Event"

sales_long[event_cols] = (
    sales_long[event_cols]
    .fillna("No Event")
)

In [10]:
# Confirm that missing values were successfully handled

sales_long[event_cols + ["sell_price"]].isnull().sum()

event_name_1    0
event_type_1    0
event_name_2    0
event_type_2    0
sell_price      0
dtype: int64

In [11]:
# Sort dataset chronologically for each product
# This is required before rolling statistics and lag features

sales_long = sales_long.sort_values(
    by=["id", "date"]
)

In [12]:
# Calculate rolling 28-day average sales
# Used for outlier detection

sales_long["rolling_mean_28"] = (
    sales_long.groupby("id")["sales"]
    .transform(lambda x: x.rolling(28).mean())
)

# Calculate rolling 28-day standard deviation
# Measures sales variability

sales_long["rolling_std_28"] = (
    sales_long.groupby("id")["sales"]
    .transform(lambda x: x.rolling(28).std())
)

In [13]:
# Flag observations as outliers if:
# sales > rolling_mean + 3 * rolling_std

sales_long["is_outlier"] = (
    sales_long["sales"] >
    sales_long["rolling_mean_28"]
    + 3 * sales_long["rolling_std_28"]
)

In [14]:
# Count how many records are classified as outliers

sales_long["is_outlier"].value_counts()

is_outlier
False    57549251
True       778119
Name: count, dtype: int64

In [15]:
# List categorical variables to encode

categorical_cols = [
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2"
]

In [16]:
# Initialize dictionary to store encoders

encoders = {}

# Apply Label Encoding to each categorical column

for col in categorical_cols:
    
    # Create encoder
    le = LabelEncoder()
    
    # Transform categorical values into numeric labels
    sales_long[col] = le.fit_transform(
        sales_long[col].astype(str)
    )
    
    # Save encoder for future inverse transformation
    encoders[col] = le

In [17]:
# Display first rows of cleaned dataset

sales_long.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,...,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,rolling_mean_28,rolling_std_28,is_outlier
1612,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_1,3,2011-01-29,11101,...,2,3,1,0,0,0,2.0,NaN,NaN,False
32102,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_2,0,2011-01-30,11101,...,2,3,1,0,0,0,2.0,NaN,NaN,False
62592,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_3,0,2011-01-31,11101,...,2,3,1,0,0,0,2.0,NaN,NaN,False
93082,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_4,1,2011-02-01,11101,...,2,3,1,1,1,0,2.0,NaN,NaN,False
123572,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_5,4,2011-02-02,11101,...,2,3,1,1,0,1,2.0,NaN,NaN,False


In [18]:
# Display dataset structure and column data types

sales_long.info()

<class 'pandas.core.frame.DataFrame'>
Index: 58327370 entries, 1612 to 58325932
Data columns (total 26 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               object        
 1   item_id          int64         
 2   dept_id          int64         
 3   cat_id           int64         
 4   store_id         int64         
 5   state_id         int64         
 6   day              object        
 7   sales            int64         
 8   date             datetime64[ns]
 9   wm_yr_wk         int64         
 10  weekday          object        
 11  wday             int64         
 12  month            int64         
 13  year             int64         
 14  d                object        
 15  event_name_1     int64         
 16  event_type_1     int64         
 17  event_name_2     int64         
 18  event_type_2     int64         
 19  snap_CA          int64         
 20  snap_TX          int64         
 21  snap_WI          int64         

In [19]:
# Save cleaned and preprocessed dataset
# Parquet format is efficient for large datasets

sales_long.to_parquet(
    "../data/processed/preprocessed_data.parquet",
    index=False
)